# Test Synthethic Data Generation (SDG) AUG-PE Algorithm

The idea of this algorithm is to generate synthetic copies of sensitive text data using an LLM with an API without the LLM ever reading any of the real data. The algorithm also includes a Differential Privacy (DP) garuantee that ensure that the removal of any one record does not impact the distribution of the data, strenghthening the privacy protections of the algorithm. 

I think in a proper implementation I would set up two classes, one that included the public data and api access and one that had the private data in a hidden variable just to avoid an overlap.

## Step 0 - Input Variables

Generate some fake restuarant reviews for testing and some of the global AUG-PE parameters.

In [ ]:
import anthropic
import os
import numpy as np
import pandas as pd
from great_tables import GT, style, loc
import pandas as pd
from sentence_transformers import SentenceTransformer

In [46]:
# Restaurant reviews
private_data = [
    "The restaurant was fantastic and the service was quick.",
    "Food arrived cold and delivery was very slow.",
    "The pizza crust was so good! But the toppings were a bit disappointing.", 
    "Absolutely loved the ambiance, cozy and warm with excellent staff.",
    "Waited over an hour for my order and it was completely wrong.",
    "The pasta was perfectly cooked but the portion sizes were tiny.",
    "Best burger I have ever had, will definitely be coming back!",
    "The sushi was fresh but the rice was overcooked and mushy.",
    "Rude staff and overpriced food, not worth the trip at all.",
    "Desserts were incredible, the chocolate lava cake was divine."
]

# Global Varibales

N_SYNTHETIC = 10

EPSILON = 4.0        # less noise, paper shows this still gives good privacy
N_CANDIDATES = 15    # more diverse initial pool
T = 5                # more iterations to spread out


In [47]:
# Connect to anthropic API and load the encoding model
client = anthropic.Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY"))
encoder = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3995.94it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


## Step 1 - DP_NN_HISTOGRAM

DP_NN_HISTOGRAM constructs a histogram of the nearest neighbours of the real data compared to the synthethic data. Distance is based on the embedding model $\Phi$. Then random gaussian noise $\frac{1}{\epsilon}$ is added to enforce DP privacy. This histogram is then used to draw samples from.

At the most basic level, it is just counting for each synthethic record how many real records is the synthetic record the most similar to. Then we draw from this distribution to get a distribution that should match the distribution of the real data.

In [48]:
# --- DP-NN HISTOGRAM ---
def dp_nn_histogram(candidates, private_texts, epsilon, n_resample):
    priv_embs = encoder.encode(private_texts, normalize_embeddings=True)
    cand_embs = encoder.encode(candidates, normalize_embeddings=True)

    counts = np.zeros(len(candidates))
    for emb in priv_embs:
        best_idx = int(np.argmax(cand_embs @ emb))
        counts[best_idx] += 1

    print(f"  Raw counts:   {counts.astype(int).tolist()}")
    noisy_counts = counts + np.random.laplace(0, 1.0 / epsilon, size=counts.shape)
    
    # Clip negatives and normalise to a probability distribution
    noisy_counts = np.clip(noisy_counts, 0, None)
    probs = noisy_counts / noisy_counts.sum()
    print(f"  Resample probs: {[f'{p:.2f}' for p in probs]}")

    # Draw n_resample candidates from the distribution
    chosen_indices = np.random.multinomial(n_resample, probs)
    resampled = []
    for idx, count in enumerate(chosen_indices):
        resampled.extend([candidates[idx]] * count)

    print(f"  Resampled pool ({n_resample}): {resampled}")
    return resampled

# Step 2 - RANDOM_API

The random api uses a prompt to draw initial candidates from the LLM API. In future, some additional subcategories should be enforced for better initital draws. 

In [49]:
# --- RANDOM API ---
def random_api(n):
    print(f"\n[RANDOM API] Generating {n} initial candidates...")
    prompt = (
        f"Generate {n} restaurant reviews: roughly 1/3 positive, 1/3 negative, 1/3 mixed sentiment. "
        "Each review should be 1-2 sentences. "
        "Return ONLY a numbered list, one review per line."
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    candidates = candidates[:n]
    for i, c in enumerate(candidates, 1):
        print(f"  {i}. {c}")
    return candidates

## Step 3 - VARIATION API

The VARIATION_API is responsible for drawing a sample from the DP_NN_HISTOGRAM, and then using the LLM API to add some variation to the drawn set of samples. This just helps to add some additional variation to the samples.



In [50]:
# --- VARIATION API ---
# VARIATION_API now takes the full resampled pool
def variation_api(text, n):
    prompt = (
        f"Generate {n} diverse variations of the following restaurant review, "
        "each with the same general sentiment but different wording and style. "
        "Return ONLY a numbered list, one review per line.\n\n"
        f"Review: {text}"
    )
    response = client.messages.create(
        model="claude-sonnet-4-20250514",
        max_tokens=512,
        messages=[{"role": "user", "content": prompt}]
    )
    lines = [l.strip() for l in response.content[0].text.strip().split("\n") if l.strip()]
    candidates = []
    for line in lines:
        parts = line.split('. ', 1)
        candidates.append(parts[1].strip() if len(parts) == 2 and parts[0].isdigit() else line)
    return candidates[:n]

## Step 4 - Run the Loop

Generates random reviews, finds the most similar records, adds variation to them, normally repeats for a certain number of runs. 

In [51]:
# Draw initial candidates
candidates = random_api(N_CANDIDATES)

for t in range(T):
    print(f"\n{'='*60}\nITERATION T={t+1}\n{'='*60}")
    resampled = dp_nn_histogram(candidates, private_data, EPSILON, n_resample=N_CANDIDATES)
    
    print(f"\n[VARIATION API]")
    candidates = []
    for text in resampled:
        varied = variation_api([text], n=1)
        candidates.extend(varied)
        print(f"  - {varied[0]}")

# Final synthetic dataset = last iteration's candidates
synthetic_data = candidates


[RANDOM API] Generating 15 initial candidates...
  1. The pasta was perfectly al dente and the service was exceptional - definitely coming back next week!
  2. Waited 45 minutes for cold food and the staff seemed completely overwhelmed and unprofessional.
  3. The atmosphere is lovely and the appetizers were great, but my main course was disappointingly bland.
  4. Outstanding flavors, creative presentation, and our server made excellent wine recommendations throughout the meal.
  5. Overpriced mediocre food with a dirty dining room - I've had better meals at fast food chains.
  6. Beautiful decor and friendly staff, though the steaks were overcooked and arrived lukewarm.
  7. Fresh ingredients, generous portions, and the chef even came out to check on our table personally.
  8. The worst dining experience I've had in years - rude hostess, terrible food, and they got our order completely wrong.
  9. Great cocktails and the dessert was amazing, but the entrees took forever and weren't 

## Step 5 - Display results

Note the algorithm produces a distribution of synthetic data, but each one isn't matched to it's closest real 

In [ ]:
# --- RESULTS TABLE ---
def render_table(private_data, synthetic_data):
    real_embs = encoder.encode(private_data, normalize_embeddings=True)
    syn_embs  = encoder.encode(synthetic_data, normalize_embeddings=True)
    sim_matrix = real_embs @ syn_embs.T

    used = set()
    rows = []
    for i in range(len(private_data)):
        sims = sim_matrix[i].copy()
        for j in used:
            sims[j] = -999
        best_j = int(np.argmax(sims))
        used.add(best_j)
        best_sim = sim_matrix[i, best_j]
        conf = "High" if best_sim >= 0.8 else "Medium" if best_sim >= 0.6 else "Low"
        rows.append({
            "Real Review":        private_data[i],
            "Synthetic Match":    synthetic_data[best_j],
            "Cosine Sim":         round(float(best_sim), 3),
            "Confidence":         conf
        })

    df = pd.DataFrame(rows)

    tbl = (
        GT(df)
        .tab_header(
            title="AUG-PE Results",
            subtitle=f"DP ε={EPSILON} — Budget spent: {EPSILON * T:.2f} ({T} iterations)"
        )
        .tab_style(
            style=style.fill(color="#d4edda"),
            locations=loc.body(rows=df.index[df["Confidence"] == "High"].tolist())
        )
        .tab_style(
            style=style.fill(color="#fff3cd"),
            locations=loc.body(rows=df.index[df["Confidence"] == "Medium"].tolist())
        )
        .tab_style(
            style=style.fill(color="#f8d7da"),
            locations=loc.body(rows=df.index[df["Confidence"] == "Low"].tolist())
        )
        .fmt_number(columns="Cosine Sim", decimals=3)
        .cols_width({"Real Review": "35%", "Synthetic Match": "35%", "Cosine Sim": "15%", "Confidence": "15%"})
    )
    return tbl

# At the end of the script:
render_table(private_data, candidates)

╭────────────────────────────────────────────────────┬────────────────────────────────────────────────────┬──────────────┬──────────────╮
│ Real Review                                        │ Best Synthetic Match                               │   Cosine Sim │ Confidence   │
├────────────────────────────────────────────────────┼────────────────────────────────────────────────────┼──────────────┼──────────────┤
│ The restaurant was fantastic and the service was   │ This restaurant was an absolute nightmare from     │        0.673 │ Medium       │
│ quick.                                             │ start to fini                                      │              │              │
│ Food arrived cold and delivery was very slow.      │ This restaurant was an absolute nightmare from     │        0.499 │ Low          │
│                                                    │ start to fini                                      │              │              │
│ The pizza crust was so good! But

## Improvements

- This was put together really quickly, need to refer to the [paper](https://alphapav.github.io/augpe-dpapitext/) or better use their code. There are defintely issues.
- Draw multiples Synethtic samples per real dataset to get a better distribution
- Add some tests on a benchmark like a sentiment analysis.